# GFC (Global Financial Crisis) Analysis: Pre/Post-2008 Split

The 2008 Global Financial Crisis marked a sharp shift in the political economy of the OECD: austerity programmes, deleveraging, retreat from some aspects of financial globalisation, and — in some countries — expansion of welfare spending to cushion the shock. That combination makes the GFC a natural structural-break candidate for our model.

This notebook focuses specifically on the GFC split (1980–2007 vs. 2008–2023) and asks three questions:

1. **Does the baseline globalisation → welfare relationship look different in the two eras?**
2. **Is the split robust to the choice of standard errors?** (Driscoll-Kraay vs. two-way clustering — the Pesaran CD and Modified Wald tests both reject independence for this panel.)
3. **Is there a formal structural break at 2008?** Chow test, rolling coefficient plot.

Sibling notebook `03_se_comparison.ipynb` shows the SE sensitivity for *all four* eras (pre/post China shock + pre/post GFC) side by side — this notebook zooms in on the GFC split.

## Setup

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path("..").resolve()
sys.path.insert(0, str(REPO_ROOT / "src"))

import matplotlib.pyplot as plt
import pandas as pd
from linearmodels.panel import compare

from analysis.regression_utils import (
    LATEX_LABEL_MAP,
    prepare_regression_data,
    run_panel_ols,
    significance_stars,
)
from clean.panel_utils import add_welfare_regimes, create_lags
from clean.structural_breaks import (
    bai_perron_test,
    chow_test,
    plot_bai_perron_bic,
    rolling_ols_coefficients,
)
from clean.utils import load_config

config = load_config()
master = add_welfare_regimes(
    pd.read_parquet(REPO_ROOT / "data" / "final" / "master_dataset.parquet")
)

indices = config.get("indices", ["KOFGI", "KOFEcGI", "KOFSoGI", "KOFPoGI"])
ctrls = config.get(
    "controls",
    ["ln_gdppc", "inflation_cpi", "deficit", "debt", "ln_population", "dependency_ratio"],
)
dep_var = config.get("dependent_var", "sstran")

PRE_GFC = (1980, 2007)
POST_GFC = (2008, 2023)
GFC_TABLES_DIR = REPO_ROOT / "outputs" / "tables" / "gfc"
GFC_TABLES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Master panel: {master.shape[0]} rows, {master['iso3'].nunique()} countries, "
      f"{master['year'].min()}-{master['year'].max()}")
print(f"Pre-GFC window:  {PRE_GFC[0]}-{PRE_GFC[1]}")
print(f"Post-GFC window: {POST_GFC[0]}-{POST_GFC[1]}")

## Helper: run baseline per-index for a year window

Creates lags on the full panel (so the first year in the window still has a valid lag), then filters by year, runs PanelOLS with entity + time fixed effects. Two SE flavours so we can show both.

In [ ]:
def fit_window(df, idx_name, start, end, cov_type="clustered"):
    reg_data = create_lags(df, [idx_name] + ctrls, lags=[1])
    window = reg_data[(reg_data["year"] >= start) & (reg_data["year"] <= end)].copy()
    indep = f"{idx_name}_lag1"
    lagged_ctrls = [f"{v}_lag1" for v in ctrls]
    ols_data, exog = prepare_regression_data(
        window, dep_var, indep, lagged_ctrls, interactions=False
    )
    if cov_type == "kernel":
        return run_panel_ols(ols_data, dep_var, exog, cov_type="kernel")
    return run_panel_ols(ols_data, dep_var, exog)  # default: two-way clustering


def _summarise(res, indep_var):
    return {
        "Coef": round(float(res.params[indep_var]), 4),
        "SE": round(float(res.std_errors[indep_var]), 4),
        "t-stat": round(float(res.tstats[indep_var]), 2),
        "p-value": round(float(res.pvalues[indep_var]), 4),
        "Stars": significance_stars(res.pvalues[indep_var]),
        "N": int(res.nobs),
    }

## Part 1: Baseline pre/post-GFC comparison (two-way clustering)

Runs the same specification — `sstran ~ KOF_{t-1} + controls_{t-1}` with entity + time fixed effects — separately on the pre-GFC and post-GFC panels. If globalisation's effect on social spending shifted after the crisis, the coefficients should differ.

In [ ]:
rows = []
pre_models = {}
post_models = {}
for idx_name in indices:
    indep = f"{idx_name}_lag1"
    pre = fit_window(master, idx_name, *PRE_GFC)
    post = fit_window(master, idx_name, *POST_GFC)
    pre_models[idx_name] = pre
    post_models[idx_name] = post
    rows.append({"Index": idx_name, "Era": "pre-GFC", **_summarise(pre, indep)})
    rows.append({"Index": idx_name, "Era": "post-GFC", **_summarise(post, indep)})

gfc_summary = pd.DataFrame(rows)
gfc_summary

In [ ]:
# Full side-by-side LaTeX summary across indices, one per era
for label, models in [("pre_gfc", pre_models), ("post_gfc", post_models)]:
    cmp = compare(models, stars=True)
    latex_str = cmp.summary.as_latex()
    for old, new in LATEX_LABEL_MAP.items():
        latex_str = latex_str.replace(old, new)
    out_path = GFC_TABLES_DIR / f"baseline_regressions_{label}.tex"
    out_path.write_text(latex_str, encoding="utf-8")
    print(f"Saved: {out_path}")

## Part 2: Driscoll-Kraay version of the pre/post-GFC comparison

The Pesaran CD test rejects cross-sectional independence of the residuals, and the Modified Wald test rejects homoskedasticity across countries — both point to DK kernel SEs being the defensible default. Point estimates are unchanged; only the standard errors (and hence stars) shift.

In [ ]:
rows = []
for idx_name in indices:
    indep = f"{idx_name}_lag1"
    pre_dk = fit_window(master, idx_name, *PRE_GFC, cov_type="kernel")
    post_dk = fit_window(master, idx_name, *POST_GFC, cov_type="kernel")
    rows.append({"Index": idx_name, "Era": "pre-GFC", **_summarise(pre_dk, indep)})
    rows.append({"Index": idx_name, "Era": "post-GFC", **_summarise(post_dk, indep)})

gfc_dk_summary = pd.DataFrame(rows)
gfc_dk_summary

## Part 3: Heterogeneity pre/post-GFC (welfare-regime interactions)

Does the GFC shift hit all welfare regimes equally, or does the pre/post difference concentrate in liberal / conservative / Mediterranean / post-communist countries (with social-democratic as reference)?

In [ ]:
def fit_window_interactions(df, idx_name, start, end):
    reg_data = create_lags(df, [idx_name] + ctrls, lags=[1])
    window = reg_data[(reg_data["year"] >= start) & (reg_data["year"] <= end)].copy()
    indep = f"{idx_name}_lag1"
    lagged_ctrls = [f"{v}_lag1" for v in ctrls]
    ols_data, exog = prepare_regression_data(
        window, dep_var, indep, lagged_ctrls, interactions=True
    )
    return run_panel_ols(ols_data, dep_var, exog)


pre_het = {idx_name: fit_window_interactions(master, idx_name, *PRE_GFC) for idx_name in indices}
post_het = {idx_name: fit_window_interactions(master, idx_name, *POST_GFC) for idx_name in indices}

for label, models in [("pre_gfc", pre_het), ("post_gfc", post_het)]:
    cmp = compare(models, stars=True)
    latex_str = cmp.summary.as_latex()
    for old, new in LATEX_LABEL_MAP.items():
        latex_str = latex_str.replace(old, new)
    out_path = GFC_TABLES_DIR / f"heterogeneity_regressions_{label}.tex"
    out_path.write_text(latex_str, encoding="utf-8")
    print(f"Saved: {out_path}")

print()
print("=== Pre-GFC heterogeneity ===")
print(compare(pre_het, stars=True))

In [ ]:
print("=== Post-GFC heterogeneity ===")
print(compare(post_het, stars=True))

## Part 4: Formal structural-break test at 2008

`chow_test` with `break_year=2008` tests H₀: coefficients are identical pre/post-GFC. The reusable helper in `src/clean/structural_breaks.py` demeans the panel first, then runs OLS in each sub-sample and F-tests slope equality.

In [ ]:
chow_rows = []
for idx_name in indices:
    result = chow_test(
        master,
        dep_var=dep_var,
        indep_var=idx_name,
        controls=ctrls,
        break_year=2008,
    )
    chow_rows.append({
        "Index": idx_name,
        "F-stat": round(result["f_stat"], 3),
        "p-value": round(result["p_value"], 4),
        "N pre": result["n_pre"],
        "N post": result["n_post"],
        "Verdict": result["verdict"],
    })

chow_df = pd.DataFrame(chow_rows)
chow_df

## Part 5: Rolling coefficient (visualising the break)

A 10-year rolling-window OLS coefficient on the lagged KOFGI. If the GFC genuinely shifts the relationship, the coefficient path should visibly drop or jump around 2008–2012.

In [ ]:
rolling = rolling_ols_coefficients(
    master,
    dep_var=dep_var,
    indep_var="KOFGI",
    controls=ctrls,
    window=10,
)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(rolling["year_end"], rolling["coef"], color="#1D4ED8", label="10y rolling coef")
ax.fill_between(
    rolling["year_end"], rolling["ci_lo"], rolling["ci_hi"],
    color="#3B82F6", alpha=0.2, label="95% CI",
)
ax.axhline(0, color="#EF4444", linestyle="-", linewidth=1, alpha=0.6)
ax.axvline(2008, color="#111827", linestyle="--", linewidth=1.2, label="2008 (GFC)")
ax.set_title("Rolling OLS coefficient of KOFGI on sstran (window = 10 years)")
ax.set_xlabel("Window end year")
ax.set_ylabel("Coefficient on KOFGI")
ax.legend()
plt.tight_layout()
plt.show()

## Part 6: Bai-Perron multiple structural break test

The Chow test (Part 4) tests for a *single* break at a known date. But what if the globalisation-welfare relationship shifted more than once — for instance around both China's WTO accession (~2001) *and* the GFC (2008)?

The **Bai-Perron (1998, 2003)** procedure answers this by searching for **multiple unknown break dates** sequentially:

1. Apply the sup-F (QLR) test to the full sample. If significant, record the break date.
2. Split at that break and search each resulting sub-sample for a further break.
3. Repeat until no segment yields a significant sup-F or `max_breaks` is reached.

The optimal number of breaks *k** is confirmed via BIC:

$$\text{BIC}(m) = n \ln\!\bigl(\text{RSS}_m / n\bigr) + (m+1)\, k \ln n$$

where *m* is the number of breaks and *k* the number of regressors. The model with the lowest BIC is preferred.

In [ ]:
bp_rows = []
bp_results = {}
for idx_name in indices:
    result = bai_perron_test(
        master,
        dep_var=dep_var,
        indep_var=idx_name,
        controls=ctrls,
        max_breaks=5,
        significance=0.05,
    )
    bp_results[idx_name] = result
    breaks_str = ", ".join(str(y) for y in result["break_years"]) if result["break_years"] else "None"
    f_str = "; ".join(f"{f:.2f}" for f in result["sup_f_stats"]) if result["sup_f_stats"] else "---"
    bp_rows.append({
        "Index": idx_name,
        "Breaks found": breaks_str,
        "Sup-F values": f_str,
        "k* (BIC)": result["k_star_bic"],
        "N": result["n_obs"],
    })

bp_df = pd.DataFrame(bp_rows)
print("Bai-Perron Sequential Multiple Break Test (5% significance)")
print(f"CV used: Andrews sup-F 5% = {result['cv_used']:.2f}\n")
bp_df

In [ ]:
# BIC plots for each index + LaTeX export
fig_dir = REPO_ROOT / "outputs" / "figures" / "gfc"
fig_dir.mkdir(parents=True, exist_ok=True)

for idx_name, res in bp_results.items():
    if res["bic_by_k"]:
        plot_bai_perron_bic(res["bic_by_k"], idx_name, res["k_star_bic"], out_dir=fig_dir)

# Show KOFGI BIC inline
kofgi_bic = bp_results.get("KOFGI", {}).get("bic_by_k", {})
if kofgi_bic:
    ks = sorted(kofgi_bic)
    bics = [kofgi_bic[m] for m in ks]
    k_star = bp_results["KOFGI"]["k_star_bic"]

    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.plot(ks, bics, "o-", color="#1D4ED8", linewidth=2, markersize=8)
    ax.axvline(k_star, color="#10B981", linestyle="--", linewidth=1.5, label=f"k* = {k_star}")
    ax.set_xticks(ks)
    ax.set_xlabel("Number of breaks (m)")
    ax.set_ylabel("BIC")
    ax.set_title("Bai-Perron BIC — KOFGI (Overall Globalization)")
    ax.legend()
    plt.tight_layout()
    plt.show()

# Export LaTeX summary
from clean.structural_breaks import _bai_perron_results_to_latex
_bai_perron_results_to_latex(list(bp_results.values()), GFC_TABLES_DIR)
print(f"Saved: {GFC_TABLES_DIR / 'bai_perron_test_table.tex'}")

## Summary

- **Baseline pre/post-GFC comparison** — Part 1 shows how the lagged globalisation coefficient moves across the break for each of the four KOF indices.
- **SE sensitivity** — the Driscoll-Kraay version in Part 2 answers whether the significance pattern survives a CSD-robust covariance estimator.
- **Heterogeneity** — Part 3 decomposes the average effect across welfare regimes so we can see whether specific country clusters drive any observed shift.
- **Formal Chow test** — Part 4 tests whether the slope coefficients are statistically different pre/post-2008.
- **Rolling coefficient** — Part 5 visualises the coefficient's drift year by year, which is informative even when the Chow test can't localise a specific break.
- **Bai-Perron multiple breaks** — Part 6 applies the sequential Bai-Perron (1998, 2003) procedure to test for *multiple* structural breaks. BIC selects the optimal number of breaks, confirming whether the GFC is the only shift or one of several.

LaTeX tables are written to `outputs/tables/gfc/` so they sit alongside the other subperiod outputs without overwriting them.